# Chunking

In diesem Step geht es darum, die Chunks für die Datenbank vorzubereiten. Da die Absätze bereits separate Elemente und die technischen Daten ohnehin nicht lange sind, bleibt hier nur das Flatening. Und man könnte die Daten in die Form der Datenbank kommen.

- DB: ChromaDB
- Input: products_enriched.json
- Output: products_chunked.jsonl

Die Struktur der JSONLs soll am Ende so aussehen wie das add() der DB erwartet.

```python
collection.add(
    documents=[...]
    metadatas=[...]
    ids=[...]
)
```

In [20]:
import json
import pandas as pd

with open('../data/processed/products_enriched.jsonl', 'r', encoding='utf-8') as f:
    products_enriched = [json.loads(line) for line in f]

In [21]:
# Chunks erstellen
chunks = []

for product in products_enriched:
    for i, desc in enumerate(product['descriptions']):
        chunks.append({
            'id':f"{product['id']}_desc_{i:02d}",
            'document': desc,
            'metadata': {
                'product_id': product['id'],
                'title': product['title'],
                'title': product['category'],
                'delivery': product['delivery_info'],
                'chunk_type': 'desc'
            }
        })
    
    for i, spec in enumerate(product['specs']):
        chunks.append({
            'id': f"{product['id']}_spec_{i:02d}",
            'document': spec['natural_language_description'],
            'metadata': {
                'product_id': product['id'],
                'title': product['title'],
                'title': product['category'],
                'delivery': product['delivery_info'],
                'chunk_type': 'spec'
            }
        })

In [22]:
with open('../data/processed/products_chunked.jsonl', 'w', encoding='utf-8') as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

# Evaluation

In [23]:
df = pd.read_json('../data/processed/products_chunked.jsonl', lines=True)
df['chunk_type'] = df['metadata'].apply(pd.Series)['chunk_type']

# Infos
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Filter
specs_df = df[df['chunk_type'] == 'spec']
descs_df = df[df['chunk_type'] == 'desc']

print(specs_df)

Shape: (1429, 4)
Columns: ['id', 'document', 'metadata', 'chunk_type']
                                                     id  \
4     Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_s...   
5     Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_s...   
6     Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_s...   
7     Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_s...   
8     Kirsch-LABO-288-PRO-ACTIVE-Laborkuehlschrank_s...   
...                                                 ...   
1424          SUFsg-3501-MediLine-400C-bis-860C_spec_33   
1425          SUFsg-3501-MediLine-400C-bis-860C_spec_34   
1426          SUFsg-3501-MediLine-400C-bis-860C_spec_35   
1427          SUFsg-3501-MediLine-400C-bis-860C_spec_36   
1428          SUFsg-3501-MediLine-400C-bis-860C_spec_37   

                                               document  \
4     Kirsch LABO-288 PRO-ACTIVE: Außenmaße 67 x 72 ...   
5     Kirsch LABO-288 PRO-ACTIVE: Außenmaße bei 90° ...   
6     Kirsch LABO-288 PRO-ACTIVE: Innenmaße